# AI3 – Machine Project
## Jherrie Nathaniel C. Millena
## Xabina Rein Teodoro
## Pratik Nanwani
## CSS181 Section 03 | Group: Kamote
### RAG-Based LLM Chatbot using Qwen2.5-1.5B-Instruct

---

> **Submission:** `AI3-Project-CSS181_03-GroupKamote.zip`  
> **Deadline:** March 05, 2026 | **Demo:** March 10–12, 2026

In [ ]:
!pip install -q transformers accelerate bitsandbytes sentence-transformers faiss-cpu pypdf gradio rouge-score nltk

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 69.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.4/331.4 kB 36.6 MB/s eta 0:00:00


In [ ]:
import torch
import numpy as np
import time
import re
import json
import warnings
warnings.filterwarnings("ignore")

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer
from pypdf import PdfReader
import faiss
import nltk
from rouge_score import rouge_scorer as rs_module
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print("All libraries imported.")
print(f"CUDA available: {torch.cuda.is_available()}")

All libraries imported.
CUDA available: True


In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

print(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

print("Loading model with 4-bit quantization...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=bnb_config
)
model.eval()
print(f"Model loaded: {MODEL_NAME}")

Loading tokenizer: Qwen/Qwen2.5-1.5B-Instruct


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Loading model with 4-bit quantization...


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded: Qwen/Qwen2.5-1.5B-Instruct


In [ ]:
def load_and_clean_pdf(pdf_path: str) -> str:
    reader = PdfReader(pdf_path)
    full_text = ""
    for page in reader.pages:
        raw = page.extract_text() or ""
        cleaned = re.sub(r'\n{3,}', '\n\n', raw)
        cleaned = re.sub(r'[ \t]+', ' ', cleaned)
        full_text += cleaned + "\n"
    return full_text


def split_text_with_overlap(text: str, chunk_size: int = 500, overlap: int = 100) -> list:
    chunks = []
    start = 0
    chunk_id = 0
    while start < len(text):
        chunk_text = text[start:start + chunk_size]
        if chunk_text.strip():
            chunks.append({"chunk_id": chunk_id, "text": chunk_text})
            chunk_id += 1
        start += chunk_size - overlap
    return chunks


PDF_PATH = "/content/seaman.pdf"
pdf_text    = load_and_clean_pdf(PDF_PATH)
chunks      = split_text_with_overlap(pdf_text, chunk_size=500, overlap=100)
chunk_texts = [c["text"] for c in chunks]

print(f"Total characters : {len(pdf_text):,}")
print(f"Total chunks     : {len(chunks)}")

Total characters : 546,648
Total chunks     : 1367


In [ ]:
EMBED_MODEL_NAME = "all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMBED_MODEL_NAME)

print("Generating embeddings...")
chunk_embeddings = embedder.encode(chunk_texts, show_progress_bar=True, batch_size=64)

dimension   = chunk_embeddings.shape[1]
faiss_index = faiss.IndexFlatL2(dimension)
faiss_index.add(np.array(chunk_embeddings, dtype=np.float32))

print(f"Embedding dim     : {dimension}")
print(f"Vectors in index  : {faiss_index.ntotal}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generating embeddings...


Batches:   0%|          | 0/22 [00:00<?, ?it/s]

Embedding dim     : 384
Vectors in index  : 1367


In [ ]:
def retrieve_context(query: str, top_k: int = 3) -> tuple:
    query_embedding = embedder.encode([query], convert_to_numpy=True)
    distances, indices = faiss_index.search(query_embedding.astype(np.float32), top_k)
    retrieved_chunks = [chunk_texts[i] for i in indices[0] if i < len(chunk_texts)]
    context_str = "\n\n---\n\n".join(retrieved_chunks)
    return context_str, retrieved_chunks, distances[0]


SYSTEM_PROMPT = (
    "You are an expert maritime labor law assistant. "
    "Answer the user's question ONLY using the provided context from official documents. "
    "If the context does not contain enough information to answer, say: "
    "'I could not find a specific answer in the available documents. "
    "Please consult the official POEA guidelines.' "
    "Do not make up or infer information not present in the context. Be concise and accurate."
)


def generate_answer(query: str, top_k: int = 3) -> dict:
    if not query or not query.strip():
        return {"answer": "Please enter a valid question.", "context": "", "chunks": [], "latency_sec": 0.0, "distances": []}

    t_start = time.time()

    try:
        context, retrieved_chunks, distances = retrieve_context(query, top_k=top_k)
    except Exception as e:
        return {"answer": f"Retrieval error: {str(e)}", "context": "", "chunks": [], "latency_sec": 0.0, "distances": []}

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Context from official documents:\n---\n{context}\n---\n\nQuestion: {query}\n\nAnswer based only on the context above:"}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(model.device)

    try:
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=350,
                temperature=0.3,
                do_sample=True,
                top_p=0.9,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.eos_token_id
            )
    except Exception as e:
        return {"answer": f"Generation error: {str(e)}", "context": context, "chunks": retrieved_chunks, "latency_sec": 0.0, "distances": distances.tolist()}

    input_len = inputs["input_ids"].shape[1]
    answer    = tokenizer.decode(output_ids[0][input_len:], skip_special_tokens=True).strip()
    latency   = round(time.time() - t_start, 2)

    return {"answer": answer, "context": context, "chunks": retrieved_chunks, "latency_sec": latency, "distances": distances.tolist()}


print("Pipeline ready — Qwen2.5-1.5B-Instruct (Local 4-bit)")

Pipeline ready — Qwen2.5-1.5B-Instruct (Local 4-bit)


In [ ]:
def _cosine_sim(text_a: str, text_b: str) -> float:
    if not text_a.strip() or not text_b.strip():
        return 0.0
    emb_a = embedder.encode([text_a], convert_to_numpy=True)[0]
    emb_b = embedder.encode([text_b], convert_to_numpy=True)[0]
    dot   = np.dot(emb_a, emb_b)
    norm  = np.linalg.norm(emb_a) * np.linalg.norm(emb_b)
    return float(dot / norm) if norm > 0 else 0.0

def compute_recall_at_k(retrieved_chunks, relevant_keywords):
    if not relevant_keywords: return 1.0
    combined = " ".join(retrieved_chunks).lower()
    found = sum(1 for kw in relevant_keywords if kw.lower() in combined)
    return round(found / len(relevant_keywords), 4)

def compute_precision_at_k(retrieved_chunks, relevant_keywords):
    if not retrieved_chunks or not relevant_keywords: return 0.0
    relevant_count = sum(1 for chunk in retrieved_chunks if any(kw.lower() in chunk.lower() for kw in relevant_keywords))
    return round(relevant_count / len(retrieved_chunks), 4)

def compute_context_relevancy(query, retrieved_chunks):
    if not retrieved_chunks: return 0.0
    return round(float(np.mean([_cosine_sim(query, chunk) for chunk in retrieved_chunks])), 4)

def compute_faithfulness(answer, retrieved_chunks):
    if not answer.strip() or not retrieved_chunks: return 0.0
    return round(float(np.max([_cosine_sim(answer, chunk) for chunk in retrieved_chunks])), 4)

def compute_answer_relevancy(query, answer):
    return round(_cosine_sim(query, answer), 4)

def compute_rouge_l(hypothesis, reference):
    if not hypothesis.strip() or not reference.strip(): return 0.0
    scorer = rs_module.RougeScorer(['rougeL'], use_stemmer=True)
    return round(scorer.score(reference, hypothesis)['rougeL'].fmeasure, 4)

print("Evaluation metric functions ready.")

Evaluation metric functions ready.


In [ ]:
TEST_SET = [
    {"query": "What is the maximum duration of a seafarer's employment contract?",
     "relevant_keywords": ["contract", "employment", "period", "term", "month"],
     "reference_answer": "The standard employment contract duration for a seafarer is typically not more than 12 months."},
    {"query": "What are the medical benefits available to seafarers who get injured on board?",
     "relevant_keywords": ["medical", "sickness", "injury", "treatment", "allowance"],
     "reference_answer": "Injured seafarers are entitled to medical treatment, sickness allowance, and disability benefits."},
    {"query": "Under what conditions can a seafarer be repatriated?",
     "relevant_keywords": ["repatri", "return", "cost", "passage", "termination"],
     "reference_answer": "A seafarer may be repatriated upon completion of contract or on medical grounds."},
    {"query": "What happens if a seafarer dies during the contract?",
     "relevant_keywords": ["death", "burial", "benefit", "compensation", "family"],
     "reference_answer": "Beneficiaries are entitled to death benefits and burial assistance."},
    {"query": "What disciplinary actions can be taken against a seafarer?",
     "relevant_keywords": ["disciplin", "offense", "dismiss", "penalty", "misconduct"],
     "reference_answer": "Disciplinary actions include warnings, fines, and dismissal for serious offenses."},
    {"query": "How is the basic monthly salary of a seafarer determined?",
     "relevant_keywords": ["salary", "wage", "basic", "pay", "rate"],
     "reference_answer": "The basic monthly salary is agreed upon in the employment contract per POEA minimum wage."},
    {"query": "What are the obligations of the ship owner to the seafarer?",
     "relevant_keywords": ["owner", "employer", "obligat", "shall", "provide"],
     "reference_answer": "The ship owner must provide safe conditions, timely payment, and medical care."},
    {"query": "Is a seafarer entitled to overtime pay?",
     "relevant_keywords": ["overtime", "hours", "work", "additional", "pay"],
     "reference_answer": "Seafarers are entitled to overtime pay for hours beyond standard working hours."},
    {"query": "What insurance coverage is provided to Filipino seafarers?",
     "relevant_keywords": ["insurance", "cover", "protect", "life", "accident"],
     "reference_answer": "Filipino seafarers are covered by life and personal accident insurance."},
    {"query": "What are the grounds for termination of a seafarer's contract before its completion?",
     "relevant_keywords": ["terminat", "dismiss", "cause", "ground", "pre-terminat"],
     "reference_answer": "Pre-termination grounds include misconduct, medical unfitness, or mutual agreement."},
]
print(f"Test set loaded: {len(TEST_SET)} unseen queries.")

Test set loaded: 10 unseen queries.


In [ ]:
evaluation_results = []
MODEL_LABEL = "Qwen2.5-1.5B-Instruct (Local 4-bit)"

print(f"Running evaluation — {MODEL_LABEL}\n")
print("=" * 80)

for i, tc in enumerate(TEST_SET):
    print(f"\n[Query {i+1}/{len(TEST_SET)}] {tc['query']}")
    result  = generate_answer(tc["query"], top_k=3)
    answer  = result["answer"]
    chunks  = result["chunks"]
    latency = result["latency_sec"]

    row = {
        "query"            : tc["query"],
        "answer"           : answer,
        "reference_answer" : tc["reference_answer"],
        "recall_at_k"      : compute_recall_at_k(chunks, tc["relevant_keywords"]),
        "precision_at_k"   : compute_precision_at_k(chunks, tc["relevant_keywords"]),
        "context_relevancy": compute_context_relevancy(tc["query"], chunks),
        "faithfulness"     : compute_faithfulness(answer, chunks),
        "answer_relevancy" : compute_answer_relevancy(tc["query"], answer),
        "rouge_l"          : compute_rouge_l(answer, tc["reference_answer"]),
        "latency_sec"      : latency
    }
    evaluation_results.append(row)

    display_ans = (answer[:120] + "...") if len(answer) > 120 else answer
    print(f"  Answer      : {display_ans}")
    print(f"  Recall@3: {row['recall_at_k']:.4f} | Prec@3: {row['precision_at_k']:.4f} | CtxRel: {row['context_relevancy']:.4f} | Faith: {row['faithfulness']:.4f} | AnsRel: {row['answer_relevancy']:.4f} | ROUGE-L: {row['rouge_l']:.4f} | Lat: {latency}s")

print("\n" + "=" * 80)
print("Evaluation complete.")

Running evaluation — Qwen2.5-1.5B-Instruct (Local 4-bit)


[Query 1/10] What is the maximum duration of a seafarer's employment contract?
  Answer      : I could not find a specific answer in the available documents. Please consult the official POEA guidelines.
  Recall@3: 0.0000 | Prec@3: 0.0000 | CtxRel: 0.3738 | Faith: 0.2831 | AnsRel: 0.2364 | ROUGE-L: 0.0625 | Lat: 5.4s

[Query 2/10] What are the medical benefits available to seafarers who get injured on board?
  Answer      : The context does not provide specific details about medical benefits for seafarers. Therefore, I could not find a specif...
  Recall@3: 0.2000 | Prec@3: 0.3333 | CtxRel: 0.4841 | Faith: 0.3883 | AnsRel: 0.7142 | ROUGE-L: 0.0870 | Lat: 6.75s

[Query 3/10] Under what conditions can a seafarer be repatriated?
  Answer      : Under the conditions described, a seafarer cannot be repatriated under any circumstances as the text does not mention an...
  Recall@3: 0.0000 | Prec@3: 0.0000 | CtxRel: 0.4831 | Faith: 0.4

In [ ]:
metrics_keys = ["recall_at_k", "precision_at_k", "context_relevancy",
                "faithfulness", "answer_relevancy", "rouge_l", "latency_sec"]

print(f"\n{'=' * 55}")
print(f"AGGREGATE RESULTS — {MODEL_LABEL}")
print(f"{'=' * 55}")
for key in metrics_keys:
    print(f"  {key:<25}: {np.mean([r[key] for r in evaluation_results]):.4f}")
print(f"{'=' * 55}")

print("\nDETAILED TABLE")
print("-" * 125)
print(f"{'#':<4} {'Query (truncated)':<45} {'Recall@3':<10} {'Prec@3':<8} {'CtxRel':<8} {'Faith':<8} {'AnsRel':<8} {'ROUGE-L':<9} {'Lat(s)':<7}")
print("-" * 125)
for i, r in enumerate(evaluation_results):
    q = (r['query'][:43] + "..") if len(r['query']) > 45 else r['query']
    print(f"{i+1:<4} {q:<45} {r['recall_at_k']:<10.4f} {r['precision_at_k']:<8.4f} {r['context_relevancy']:<8.4f} {r['faithfulness']:<8.4f} {r['answer_relevancy']:<8.4f} {r['rouge_l']:<9.4f} {r['latency_sec']:<7.2f}")
print("-" * 125)
print(f"{'MEAN':<50}", end="")
for key in metrics_keys:
    print(f"{np.mean([r[key] for r in evaluation_results]):<10.4f}", end="")
print()

with open("/content/qwen_evaluation_log.json", "w") as f:
    json.dump(evaluation_results, f, indent=2)
print("\nSaved: /content/qwen_evaluation_log.json")


AGGREGATE RESULTS — Qwen2.5-1.5B-Instruct (Local 4-bit)
  recall_at_k              : 0.0600
  precision_at_k           : 0.1667
  context_relevancy        : 0.4485
  faithfulness             : 0.3466
  answer_relevancy         : 0.6237
  rouge_l                  : 0.1071
  latency_sec              : 4.7680

DETAILED TABLE
-----------------------------------------------------------------------------------------------------------------------------
#    Query (truncated)                             Recall@3   Prec@3   CtxRel   Faith    AnsRel   ROUGE-L   Lat(s) 
-----------------------------------------------------------------------------------------------------------------------------
1    What is the maximum duration of a seafarer'.. 0.0000     0.0000   0.3738   0.2831   0.2364   0.0625    5.40   
2    What are the medical benefits available to .. 0.2000     0.3333   0.4841   0.3883   0.7142   0.0870    6.75   
3    Under what conditions can a seafarer be rep.. 0.0000     0.0000   0.48

In [ ]:
while True:
    question = input("You: ").strip()
    if question.lower() == "exit":
        print("Exiting.")
        break
    if not question:
        continue
    result  = generate_answer(question, top_k=3)
    answer  = result["answer"]
    chunks  = result["chunks"]
    latency = result["latency_sec"]
    print(f"\nBot: {answer}")
    print(f"\n📊 CtxRel: {compute_context_relevancy(question, chunks):.4f} | Faithfulness: {compute_faithfulness(answer, chunks):.4f} | AnsRel: {compute_answer_relevancy(question, answer):.4f} | Latency: {latency}s\n")


Bot: The text does not provide any relevant information to answer the question "tungkol san tong seaman pdf na to". The content discusses non-resident training courses, seamanship requirements, and professional development instructions without mentioning a PDF file specifically about Seaman. Therefore, the appropriate response is:

I could not find a specific answer in the available documents. Please consult the official POEA guidelines.

📊 CtxRel: 0.4960 | Faithfulness: 0.6068 | AnsRel: 0.8171 | Latency: 7.26s


Bot: The introduction of this seaman handbook emphasizes the importance of conducting oneself honorably, courageously, and with commitment while serving in the Navy combat team. It highlights the significance of maintaining proper attire and adhering to the standards set forth in the document. The introduction underscores the value placed on professionalism and ethical behavior in one's duties.

📊 CtxRel: 0.6478 | Faithfulness: 0.6219 | AnsRel: 0.7884 | Latency: 5.98s


Bot: 